<a href="https://colab.research.google.com/github/Alex-Leo-Reeves/Blessing/blob/main/Copy_of_aimodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1) Install system dependencies and Ollama
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q pyngrok
!ollama --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
# 2) Start the Ollama daemon in the background
import os, subprocess, time, signal

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
daemon = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(8)
print('Ollama server started. Checking health...')
!curl -s http://127.0.0.1:11434/api/version || true

Ollama server started. Checking health...
{"version":"0.32.15"}

In [ ]:
# 3) Pull the requested coding-specialist model.
# This uses the exact Ollama model name requested by the user.
# If it fails due to memory, fall back to qwen2.5-coder:14b or qwen2.5:7b.
!ollama pull richardyoung/qwen2.5-coder-14b-instruct-abliterated

In [ ]:
# 4) Expose the model via ngrok (HTTP tunnel works on free accounts)
from pyngrok import ngrok
import os

# Set your ngrok auth token before creating the tunnel.
os.environ['NGROK_AUTHTOKEN'] = '3BvvlLharHRC80qzPeVYNLnbrI2_3A98xw3GArmVc4idQgdE7'

ngrok.set_auth_token(os.environ['NGROK_AUTHTOKEN'])
tunnel = ngrok.connect(11434, 'http')
print('Public tunnel:', tunnel.public_url)

Public tunnel: https://unpatterned-rozanne-emigrative.ngrok-free.dev


In [ ]:
# 5) Test the model
import json, urllib.request

payload = json.dumps({
    'model': 'richardyoung/qwen2.5-coder-14b-instruct-abliterated',
    'prompt': 'Write a Python function to validate email addresses.',
    'stream': False
}).encode()

req = urllib.request.Request(
    'http://127.0.0.1:11434/api/generate',
    data=payload,
    headers={'Content-Type': 'application/json'},
    method='POST'
)

with urllib.request.urlopen(req, timeout=120) as r:
    print(r.read().decode()[:1000])

TimeoutError: timed out

In [ ]:
!ollama list
!curl -s http://127.0.0.1:11434/api/version
!curl -s http://127.0.0.1:11434/api/tags

NAME                                                          ID              SIZE      MODIFIED      
richardyoung/qwen2.5-coder-14b-instruct-abliterated:latest    fb89eb977c11    9.0 GB    7 minutes ago    
{"version":"0.32.15"}{"models":[{"name":"richardyoung/qwen2.5-coder-14b-instruct-abliterated:latest","model":"richardyoung/qwen2.5-coder-14b-instruct-abliterated:latest","modified_at":"2026-08-22T11:08:50.086577441Z","size":8988111333,"digest":"fb89eb977c1183ba837974b4a28c6b7b99cf0f537df56cde888ff979d14da500","details":{"parent_model":"","format":"gguf","family":"qwen2","families":["qwen2"],"parameter_size":"14.8B","quantization_level":"Q4_K_M","context_length":32768,"embedding_length":5120},"capabilities":["completion"]}]}

In [ ]:
import json, urllib.request

payload = json.dumps({
    "model": "richardyoung/qwen2.5-coder-14b-instruct-abliterated:latest",
    "prompt": "Write a Python function to check if a string is a valid email address.",
    "stream": False
}).encode()

req = urllib.request.Request(
    "http://127.0.0.1:11434/api/generate",
    data=payload,
    headers={"Content-Type": "application/json"},
    method="POST"
)

with urllib.request.urlopen(req, timeout=300) as r:
    print(r.read().decode()[:2000])

{"model":"richardyoung/qwen2.5-coder-14b-instruct-abliterated:latest","created_at":"2026-08-22T11:17:49.616867792Z","response":"To check if a string is a valid email address in Python, you can use regular expressions (regex) to match the pattern of a typical email address. Here's a simple function that uses the `re` module to perform this validation:\n\n```python\nimport re\n\ndef is_valid_email(email):\n    # Define a regular expression pattern for validating an email\n    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$'\n    \n    # Use the re.match function to check if the email matches the pattern\n    if re.match(pattern, email):\n        return True\n    else:\n        return False\n\n# Example usage:\nprint(is_valid_email(\"example@example.com\"))  # Output: True\nprint(is_valid_email(\"example.com\"))          # Output: False\n```\n\n### Explanation:\n- **Pattern Breakdown**:\n  - `^[a-zA-Z0-9._%+-]+`: Matches the local part of the email (before the `@` symbol), a

In [ ]:
import json, urllib.request

payload = json.dumps({
    "model": "richardyoung/qwen2.5-coder-14b-instruct-abliterated:latest",
    "prompt": "Write a Python function to check if a string is a valid email address.",
    "stream": False
}).encode()

req = urllib.request.Request(
    "https://unpatterned-rozanne-emigrative.ngrok-free.dev/api/generate",
    data=payload,
    headers={"Content-Type": "application/json"},
    method="POST"
)

with urllib.request.urlopen(req, timeout=300) as r:
    print(r.read().decode()[:2000])

{"model":"richardyoung/qwen2.5-coder-14b-instruct-abliterated:latest","created_at":"2026-08-22T11:18:55.442063407Z","response":"Here's a Python function to validate email addresses using regular expressions:\n\n```python\nimport re\n\ndef is_valid_email(email):\n    # Define the regular expression pattern for validating an email\n    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$'\n    \n    # Use the re.match function to check if the email matches the pattern\n    if re.match(pattern, email):\n        return True\n    else:\n        return False\n\n# Test the function\nprint(is_valid_email(\"example@example.com\"))  # True\nprint(is_valid_email(\"user.name+tag+sorting@example.com\"))  # True\nprint(is_valid_email(\"user@sub.example.com\"))  # True\nprint(is_valid_email(\"user@123.123.123.123\"))  # True\nprint(is_valid_email(\"user@[IPv6:2001:db8::1]\"))  # False\nprint(is_valid_email(\"plainaddress\"))  # False\nprint(is_valid_email(\"@missingusername.com\"))  # False\